In [1]:
import os
import pandas as pd
from asset_modeling.credit import loan_portfolio, private_credit_loan_model
from data.sofr import get_sofr_data
from data.ratings import get_effective_yield

fred_api_key = os.getenv("FRED_API_KEY")

#### credit functions

##### private_credit_loan_model()
Models a single private credit loan and generates a quarterly payment schedule with amortization, interest, and fees.

**Arguments:**
- `investment_name`: Name/identifier for the investment
- `investment_date`: Date the investment/loan is initiated
- `maturity_date`: Date the loan matures
- `loan_size`: Principal amount of the loan
- `spread`: Spread in basis points
- `base_rate`: Base interest rate
- `sofr_assumption`: SOFR rate assumption
- `cash_interest_rate`: Annual cash interest rate (paid quarterly)
- `pik_interest`: Annual payment-in-kind interest rate (added to balance quarterly)
- `amortization`: Annual amortization rate applied to original loan amount
- `oid`: Original issue discount as a percentage of loan_size
- `exit_fee`: Fee as percentage of remaining balance at maturity/prepayment
- `prepayment_date`: Optional date when loan is prepaid (if None, uses maturity_date)

**Returns:** DataFrame with quarterly schedule including cash flows, balances, interest, and IRR

---

##### loan_portfolio()
Aggregates multiple loans into a portfolio summary by combining individual loan schedules and performing quarterly rollup.

**Arguments:**
- `schedule_of_investments`: DataFrame or dict with one row per loan containing all required columns for `private_credit_loan_model()`

**Returns:** DataFrame with quarterly portfolio totals (invested_amount, total_payment, remaining_balance_payment, ending_balance, beginning_balance)

---
#### Example: private_credit_loan_model() 

1) Determine Cash Rate
    - Total_Rate = SOFR + Spread = Cash Interest Rate + PIK Interest Rate
        - Example: 10 = 4 + 6 = 8 + 2
    Cash Interest Rate = SOFR + Spread - PIK Interest
        - Example: 8 = 4 + 6 - 2

2) Determine the beginning and ending balance by accounting for Amortization and PIK Interest
    Where:
        - Amortization (fixed on Par Value) = Loan Par Value * (Amortization Rate/4)
        - PIK intereest = Beginning Balance * (PIK Interest Rate/4)
        - Ending balance = Beginning Balance - Amortization + PIK Interest

3) Calculate Cash Interest
    - Ending Balance * (Cash Interest Rate/4)

4) Calculate End of Term Payments
    - Where:
        - payments are made in the last quarter when the term is over or the loan is prepaid
    - Exit Fee = Loan Par Value * Exit Fee
    - Balance Repayment = Beginning Balance at the last quarter date

5) Calculate Cash Flow
    - Where:
        - Interest and Amortization are paid quarterly
        - Balance Repayment and Exit Fees are paid in final quarter (Exit fee is based on par value)
    - Cash Payment = Interest + Amortization + Balance Repayment + Fees

In [2]:
### private_credit_loan_model()
# Example usage of the private_credit_loan_model function to create a single loan schedule
maturity_date=pd.Timestamp("2029-12-31")
rates = get_sofr_data(api_key=fred_api_key, frequency='D', end_date=maturity_date)
spreads = get_effective_yield(rating="B", api_key=fred_api_key, frequency='D', end_date=maturity_date)

loan = private_credit_loan_model(
    investment_name="Example Corp Term Loan",
    investment_date=pd.Timestamp("2024-12-31"),
    maturity_date=maturity_date,
    par_value=1_000_000,
    spread=spreads,
    base_rate='SOFR',
    sofr_rates=rates,
    # sofr_assumption=0.04,
    # sofr_floor = 0.01
    pik_interest=0.02,
    amortization=0.01,
    oid=0.02,
    exit_fee=0.02,
    prepayment_date=None,
    company_uid=1,
    investment_uid=1,
    as_of_date=pd.Timestamp("2026-03-31")
)

print(f"Loan schedule generated with {len(loan)} quarterly periods")
loan

Loan schedule generated with 21 quarterly periods


,investment_name,company_uid,investment_uid,quarter_end,par_value,original_investment,invested_amount,base_rate,sofr_rate,rate_status,...,nav,contributions,distributions,ncf,cumulative_contributions,cumulative_distributions,cumulative_ncf,tvpi,effective_duration,irr
0,Example Corp Term Loan,1,1,2024-12-31,1000000,980000.0,980000.0,SOFR,0.0449,actual,...,980000.0,-980000.0,0.000000e+00,-9.800000e+05,-980000.0,0.000000e+00,-980000.000000,1.000000,None,NaN
1,Example Corp Term Loan,1,1,2025-03-31,1000000,980000.0,0.0,SOFR,0.0441,actual,...,980886.203177,-0.0,2.685000e+04,2.685000e+04,-980000.0,2.685000e+04,-953150.000000,1.025215,3.625026,0.014245
2,Example Corp Term Loan,1,1,2025-06-30,1000000,980000.0,0.0,SOFR,0.0445,actual,...,982157.644533,-0.0,2.701113e+04,2.701113e+04,-980000.0,5.386112e+04,-926138.875000,1.061711,3.41562,0.019213
3,Example Corp Term Loan,1,1,2025-09-30,1000000,980000.0,0.0,SOFR,0.0424,actual,...,983242.927866,-0.0,2.654492e+04,2.654492e+04,-980000.0,8.040605e+04,-899593.950937,1.092832,3.209468,0.088947
4,Example Corp Term Loan,1,1,2025-12-31,1000000,980000.0,0.0,SOFR,0.0387,actual,...,984183.421702,-0.0,2.567336e+04,2.567336e+04,-980000.0,1.060794e+05,-873920.587000,1.118551,3.006519,0.094000
5,Example Corp Term Loan,1,1,2026-03-31,1000000,980000.0,0.0,SOFR,0.0368,actual,...,984939.178361,-0.0,2.525195e+04,2.525195e+04,-980000.0,1.313314e+05,-848668.641987,1.136902,2.806721,0.096936
6,Example Corp Term Loan,1,1,2026-06-30,1000000,980000.0,0.0,SOFR,0.0357,actual,...,986072.057541,-0.0,2.503092e+04,2.503092e+04,-980000.0,1.563623e+05,-823637.721796,1.168493,2.610024,0.099114
7,Example Corp Term Loan,1,1,2026-09-30,1000000,980000.0,0.0,SOFR,0.0357,actual,...,987066.779051,-0.0,2.508795e+04,2.508795e+04,-980000.0,1.814502e+05,-798549.772004,1.195116,2.416379,0.100694
8,Example Corp Term Loan,1,1,2026-12-31,1000000,980000.0,0.0,SOFR,0.0357,actual,...,988061.572801,-0.0,2.514526e+04,2.514526e+04,-980000.0,2.065955e+05,-773404.507463,1.221798,2.225737,0.101938
9,Example Corp Term Loan,1,1,2027-03-31,1000000,980000.0,0.0,SOFR,0.0357,actual,...,989056.437722,-0.0,2.520287e+04,2.520287e+04,-980000.0,2.317984e+05,-748201.641599,1.248538,2.038051,0.102947


In [6]:
schedule_of_investments = pd.DataFrame(
    {
        "investment_name": ["A Corp Term Loan", "B Corp Term Loan",  "C Corp Term Loan",  "D Corp Term Loan",  "E Corp Term Loan"],
        "company_uid": [1, 2, 3, 4, 5],
        "investment_uid": [1, 2, 3, 4, 5],
        "investment_date": [pd.Timestamp("2020-03-31"), pd.Timestamp("2020-06-30"), pd.Timestamp("2020-09-30"), pd.Timestamp("2020-12-31"), pd.Timestamp("2021-03-31")],
        "maturity_date": [pd.Timestamp("2025-06-30"), pd.Timestamp("2027-06-30"), pd.Timestamp("2026-12-31"), pd.Timestamp("2028-03-31"), pd.Timestamp("2029-06-30")],
        "par_value": [1_000_000, 1_250_000, 1_000_000, 1_050_000, 1_600_000],
        # "spread": [0.07, 0.06, 0.06, 0.08, 0.07],
        "spread": ['actual', 'actual', 'actual', 'actual', 'actual'],
        "base_rate": ['SOFR', 'SOFR', 'SOFR', 'SOFR', 'SOFR'],
        "sofr_assumption": ['actual', 'actual', 'actual', 'actual', 'actual'],
        "pik_interest": [0.02, 0.02, 0.02, 0.02, 0.02],
        "amortization": [0.01, 0.01, 0.01, 0.01, 0.01],
        "oid": [0.03, 0.02, 0.03, 0.04, 0.02],
        "exit_fee": [0.02, 0.02, 0.02, 0.02, 0.02],
        "prepayment_date": [None, None, None, None, None],
    }
)

portfolio, funds, funds_summary = loan_portfolio(schedule_of_investments, rates, spreads, as_of_date=pd.Timestamp("2026-03-31"))

In [7]:
portfolio.head()

,quarter_end,invested_amount,amortization,cash_interest,fees,remaining_balance_payment,total_payment,ending_balance,beginning_balance,nav,contributions,distributions,ncf,cumulative_contributions,cumulative_distributions,cumulative_ncf,status,irr,tvpi
0,2020-03-31,970000.0,0.0,0.000000,0.0,0.0,-9.700000e+05,1.000000e+06,1.000000e+06,970000.0,-970000.0,0.000000,-9.700000e+05,-970000.0,0.000000,-9.700000e+05,actual,None,1.0
1,2020-06-30,1225000.0,2500.0,20225.000000,0.0,0.0,-1.202275e+06,2.252500e+06,2.250000e+06,2197501.814422,-1225000.0,22725.000000,-1.202275e+06,-2195000.0,22725.000000,-2.172275e+06,actual,0.026007,1.011493
2,2020-09-30,970000.0,5625.0,36131.687500,0.0,0.0,-9.282433e+05,3.258138e+06,3.252500e+06,3170688.789324,-970000.0,41756.687500,-9.282433e+05,-3165000.0,64481.687500,-3.100518e+06,actual,0.022179,1.022171
3,2020-12-31,1008000.0,8125.0,46365.689375,0.0,0.0,-9.535093e+05,4.316303e+06,4.308138e+06,4184018.585176,-1008000.0,54490.689375,-9.535093e+05,-4173000.0,118972.376875,-4.054028e+06,actual,0.020543,1.03115
4,2021-03-31,1568000.0,10750.0,52738.259844,0.0,0.0,-1.504512e+06,5.927135e+06,5.916303e+06,5756542.037071,-1568000.0,63488.259844,-1.504512e+06,-5741000.0,182460.636719,-5.558539e+06,actual,0.077636,1.034489


In [8]:
funds.head()

,investment_name,company_uid,investment_uid,quarter_end,par_value,original_investment,invested_amount,base_rate,sofr_rate,rate_status,...,nav,contributions,distributions,ncf,cumulative_contributions,cumulative_distributions,cumulative_ncf,tvpi,effective_duration,irr
0,A Corp Term Loan,1,1,2020-03-31,1000000,970000.0,970000.0,SOFR,0.0001,actual,...,970000.0,-970000.0,0.000000,-970000.000000,-970000.0,0.000000,-970000.000000,1.000000,None,NaN
1,A Corp Term Loan,1,1,2020-06-30,1000000,970000.0,0.0,SOFR,0.0010,actual,...,972501.814422,-0.0,22725.000000,22725.000000,-970000.0,22725.000000,-947275.000000,1.054744,3.707413,0.013071
2,A Corp Term Loan,1,1,2020-09-30,1000000,970000.0,0.0,SOFR,0.0008,actual,...,974208.111584,-0.0,22725.437500,22725.437500,-970000.0,45450.437500,-924549.562500,1.089719,3.489098,0.017173
3,A Corp Term Loan,1,1,2020-12-31,1000000,970000.0,0.0,SOFR,0.0007,actual,...,976018.612574,-0.0,22751.001875,22751.001875,-970000.0,68201.439375,-901798.560625,1.129269,3.27557,0.079257
4,A Corp Term Loan,1,1,2021-03-31,1000000,970000.0,0.0,SOFR,0.0001,actual,...,977297.183701,-0.0,22650.751250,22650.751250,-970000.0,90852.190625,-879147.809375,1.152765,3.066723,0.083979


In [9]:
funds_summary

,investment_name,company_uid,investment_uid,investment_date,maturity_date,par_value,spread,base_rate,sofr_assumption,pik_interest,amortization,oid,exit_fee,prepayment_date,total_payment,irr
0,A Corp Term Loan,1,1,2020-03-31,2025-06-30,1000000,actual,SOFR,actual,0.02,0.01,0.03,0.02,None,7.412739e+05,0.141755
1,B Corp Term Loan,2,2,2020-06-30,2027-06-30,1250000,actual,SOFR,actual,0.02,0.01,0.02,0.02,None,9.741337e+05,0.109282
2,C Corp Term Loan,3,3,2020-09-30,2026-12-31,1000000,actual,SOFR,actual,0.02,0.01,0.03,0.02,None,6.487837e+05,0.102844
3,D Corp Term Loan,4,4,2020-12-31,2028-03-31,1050000,actual,SOFR,actual,0.02,0.01,0.04,0.02,None,6.942072e+05,0.090457
4,E Corp Term Loan,5,5,2021-03-31,2029-06-30,1600000,actual,SOFR,actual,0.02,0.01,0.02,0.02,None,1.200106e+06,0.089186
